In [7]:
# Don't forget to pip freeze > requirements.txt
import math
import numpy as np
import matplotlib.pyplot as plt
import random
import string
%matplotlib inline 
# Make sure plots are displayed under each cell in Jupyter Notebook

In [46]:
class FormattedList:
    def __init__(self, data):
        self.data = data
    def __repr__(self):
        return "\n".join(self.data)

In [ ]:
class Node:
    BINARY_OPERATORS = [
        "Add {+}",
        "Subtract {-}",
        "Multiply {*}",
        "Divide {/}",
        "Power {^}"
    ]
    
    UNARY_OPERATORS = [
        "Absolute Value {.abs()}",
        "Exponential {.e()}",
        "Negate {-(x)}"
    ]
    
    ACTIVATION_FUNCTIONS = [
        "Tanh {tanh()}",
        "ReLU {relu()}",
        "Threshold ReLU {threshold_relu()}",
        "PReLU {prelu()}",
        "Leaky ReLU {leaky_relu()}",
        "Sigmoid {sigmoid()}",
        "SiLU {silu()}",
        "ELU {elu()}",
        "GELU {gelu()}",
        "SELU {selu()}",
        "Softplus {softplus()}",
        "Mish {mish()}",
        "Hardswish {hardswish()}"
    ]

    @classmethod
    def _format(cls, mapping):
        result = []
        for name, items in mapping:
            result.append(f"{name}:")
            result.extend(f"  - {item}" for item in items)
        return FormattedList(result)

    @classmethod
    def get_all(cls):
        return cls._format([
            ("Binary Operators", cls.BINARY_OPERATORS),
            ("Unary Operators", cls.UNARY_OPERATORS),
            ("Activation Functions", cls.ACTIVATION_FUNCTIONS)
        ])

    @classmethod
    def get_operators(cls):
        return cls._format([
            ("Binary Operators", cls.BINARY_OPERATORS),
            ("Unary Operators", cls.UNARY_OPERATORS)
        ])

    @classmethod
    def get_binary_operators(cls):
        return cls._format([("Binary Operators", cls.BINARY_OPERATORS)])

    get_binary = get_binary_operators

    @classmethod
    def get_unary_operators(cls):
        return cls._format([("Unary Operators", cls.UNARY_OPERATORS)])

    get_unary = get_unary_operators

    @classmethod
    def get_activations(cls):
        return cls._format([("Activation Functions", cls.ACTIVATION_FUNCTIONS)])

    get_activation = get_activation_function = get_activation_functions = get_activations

    def __init__(self,data, children=(), operator=''):
        self.data = data
        self.gradient = 0
        self.children = set(children) # Remove duplicate from tuple by turning it into set, turn set into list to access children through index
        self.operator = operator
        self._backward = lambda: None

    def __repr__(self):
        return f"Node(data={self.data})"


        

    # Binary Operators

    def __add__(self, other):
        other = other if isinstance(other, Node) else Node(other) # isinstance(object, class) checks if 'object' is a 'class' 
        output = Node(self.data + other.data, children=(self,other), operator='+')

        def add_grad():
            self.gradient += output.gradient
            other.gradient += output.gradient

        output._backward = add_grad # Stores function inside an instance variable
        return output

    def __sub__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.data - other.data, children=(self,other), operator='-')

        def sub_grad():
            self.gradient += output.gradient
            other.gradient -= output.gradient

        output._backward = sub_grad
        return output

    def __mul__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.data * other.data, children=(self,other), operator='*')

        def mul_grad():
            self.gradient += other.data * output.gradient
            other.gradient += self.data * output.gradient

        output._backward = mul_grad
        return output

    def __truediv__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.data / other.data, children=(self,other), operator='/')

        def truediv_grad():
            self.gradient += (1/other.data) * output.gradient
            other.gradient += ((-self.data)/(other.data**2) * output.gradient)

        output._backward = truediv_grad
# The second _backward represents the function, it is a way to store a functon inside a variable simply without its brackets
        return output

    def __pow__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(self.data**other.data, children=(self,other), operator='^')

        def pow_grad():
            if self.data > 0:
                self.gradient += other.data*(self.data**(other.data - 1)) * output.gradient
                other.gradient += (self.data**other.data) * math.log(self.data) * output.gradient # e is default base
            elif self.data < 0 and other.data.is_integer():
                self.gradient -= other.data*(abs(self.data)**(other.data - 1)) * output.gradient
                other.gradient += (self.data**other.data) * math.log(abs(self.data)) * output.gradient
            elif self.data < 0 and not other.data.is_integer():
                raise ValueError(f"Your base, {self} is negative while your exponent, {other} is fractional, float, non-integer hence it is an imaginary number that is impossible to compute.")
            elif self.data == 0:
                if other.data >= 1:
                    self.gradient += 0
                    other.gradient += 0
                else:
                    raise ValueError(f"Your base, {self} is zero powered by your exponent, {other} is zero, you are trying to divide 0 by 0 that is impossible to compute.")

        output._backward = pow_grad
        return output

    # Reverse Operators
    def __radd__(self, other):
        return self + other # Calls __add__ function

    def __rsub__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(other.data - self.data, children=(other, self), operator='-')

        def sub_grad():
            other.gradient += output.gradient
            self.gradient -= output.gradient
        
        output._backward = sub_grad
        return output

    def __rmul__(self, other):
        return self * other # Calls __mul__ function

    def __rtruediv__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(other.data / self.data, children=(other, self), operator='/')

        def truediv_grad():
            other.gradient += (1/self.data) * output.gradient
            self.gradient += ((-other.data)/(self.data**2) * output.gradient)

        output._backward = truediv_grad
        return output

    def __rpow__(self, other):
        other = other if isinstance(other, Node) else Node(other)
        output = Node(other.data**self.data, children=(other, self), operator='^')

        def pow_grad():
            if other.data > 0:
                other.gradient += self.data*(other.data**(self.data-1)) * output.gradient
                self.gradient += (other.data**self.data) * math.log(other.data) * output.gradient # e is default base
            elif other.data < 0 and self.data.is_integer():
                other.gradient -= self.data*(abs(other.data)**(self.data - 1)) * output.gradient
                self.gradient += (other.data**self.data) * math.log(abs(other.data)) * output.gradient
            elif other.data < 0 and not self.data.is_integer():
                raise ValueError(f"Your base, {other} is negative while your exponent, {self} is fractional, float, non-integer hence it is an imaginary number that is impossible to compute.")
            elif other.data == 0:
                if self.data >= 1:
                    other.gradient += 0
                    self.gradient += 0
                else:
                    raise ValueError(f"Your base, {other} is zero powered by your exponent, {self} is zero, you are trying to divide 0 by 0 that is impossible to compute.")

        output._backward = pow_grad
        return output


    # Unary Operators

    def __abs__(self):
        output = Node(abs(self.data), children=(self,), operator='abs')

        def abs_grad():
            self.gradient += 0 if self.data == 0 else math.copysign(1, self.data) * output.gradient

        output._backward = abs_grad
        return output

    def e(self):
        output = Node(math.exp(self.data), children=(self,), operator='e')

        def e_grad():
            self.gradient += output.data * output.gradient

        output._backward = e_grad
        return output

    def __neg__(self):
        output = Node(-self.data, children=(self,), operator='neg')

        def neg_grad():
            self.gradient -= output.gradient

        output._backward = neg_grad
        return output

    
    # Activation function

    def tanh(self):
        output = Node(math.tanh(self.data), children=(self,), operator='tanh')

        def tanh_grad():
            self.gradient += (1 - output.data**2) * output.gradient

        output._backward = tanh_grad
        return output

    def relu(self, theta):
        if theta:
            return self.threshold_relu(theta)

        output = Node(max(0.0, self.data), children=(self,), operator='relu')

        def relu_grad():
            self.gradient += 1 * output.gradient if self.data > 0 else 0 

        output._backward = relu_grad
        return output

    def threshold_relu(self, theta=0):
        output = Node(self.data if self.data > 0 else 0, children=(self,), operator=f'threshold relu {theta}')

        def threshold_relu_grad():
            self.gradient += 1 * output.gradient if self.data > 0 else 0

        output._backward = threshold_relu_grad
        return output

    def prelu(self, a=0.25):
        if self.data >= 0: 
            val = self.data
            children = (self,)
            alpha_node = None
        else:
            alpha_node = Node(a, children=(), operator='alpha')
            val = alpha_node.data * self.data
            children = (self, alpha_node)
        
        output = Node(val, children=children, operator='prelu')  

        def prelu_grad():
            if self.data >= 0:
                self.gradient += 1 * output.gradient
            else:
                self.gradient += alpha_node.data * output.gradient
                alpha_node.gradient += self.data * output.gradient

        output._backward = prelu_grad
        return output

    def leakyrelu(self, a=0.01):
        output = Node(max(a*self.data, self.data), children=(self,), operator='leakyrelu')

        def leakyrelu_grad():
            self.gradient += 1 * output.gradient if self.data > 0 else a * output.gradient

        output._backward = leakyrelu_grad
        return output

    def sigmoid(self):
        output = Node(1/(1 + math.exp(-self.data)), children=(self,), operator='sigmoid')

        def sigmoid_grad():
            self.gradient += output.data*(1 - output.data) * output.gradient

        output._backward = sigmoid_grad
        return output

    def silu(self):
        output = Node(self.data/(1 + math.exp(-self.data)), children=(self,), operator='silu')

        def silu_grad():
            self.gradient += (1/(1 + math.exp(-self.data)) + self.data*(1/(1 + math.exp(-self.data)))*(1 - 1/(1 + math.exp(-self.data)))) * output.gradient

        output._backward = silu_grad
        return output

    def elu(self, a=1):
        output = Node(self.data if self.data > 0 else a*(math.exp(self.data) - 1), children=(self,), operator='elu')

        def elu_grad():
            self.gradient += 1 * output.gradient if self.data > 0 else (output.data + a) * output.gradient 

        output._backward = elu_grad
        return output

    def gelu(self):
        output = Node((0.5*self.data)*(1 + math.erf(self.data/2**0.5)), children=(self,), operator='gelu')

        def gelu_grad():
            cdf = 0.5 * (1 + math.erf(self.data/2**0.5)) # Cumalative Distribution Function
            pdf = (1/(2*math.pi)**0.5) * math.exp(-(self.data**2)/2) # Standard Normal Probability Function
            self.gradient += (cdf + self.data*pdf) * output.gradient
        output._backward = gelu_grad
        return output

    def selu(self, lam=1.0507 , a=1.67326):
        output = Node(lam*self.data if self.data > 0 else (lam*a*math.exp(self.data) - a), children=(self,), operator='selu')

        def selu_grad():
            self.gradient += lam * output.gradient if self.data > 0 else (output.data + lam*a) * output.gradient

        output._backward = selu_grad
        return output

    def softplus(self):
        output = Node(math.log(1 + math.exp(self.data)), children=(self,), operator='softplus')

        def softplus_grad():
            self.gradient += (1/(1 + math.exp(-self.data))) * output.gradient

        output._backward = softplus_grad
        return output

    def mish(self):
        output = Node(self.data*math.tanh(math.log(1 + math.exp(self.data))), children=(self,), operator='mish')

        def mish_grad():
            self.gradient += (math.tanh(math.log(1 + math.exp(self.data))) + self.data*(1 - (math.tanh(math.log(1 + math.exp(self.data))))**2)*(1/(1 + math.exp(-self.data)))) * output.gradient

        output._backward = mish_grad
        return output

    def hardswish(self):
        output = Node(self.data * (min(max(0, self.data + 3), 6)/6), children=(self,), operator='hardswish')

        def hardswish_grad():
            if self.data < -3:
                self.gradient += 0
            elif -3 <= self.data <= 3:
                self.gradient += ((2*self.data + 3)/3) * output.gradient
            elif self.data > 3:
                self.gradient += 1 * output.gradient

        output._backward = hardswish_grad
        return output

    def backward(self):
        self.gradient = 1
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        for node in reversed(topo):
            if node._backward:
                node._backward()



In [51]:
Node.get_activation_functions()

Activation Functions:
  - Tanh {tanh()}
  - ReLU {relu()}
  - Threshold ReLU {threshold_relu()}
  - PReLU {prelu()}
  - Leaky ReLU {leaky_relu()}
  - Sigmoid {sigmoid()}
  - SiLU {silu()}
  - ELU {elu()}
  - GELU {gelu()}
  - SELU {selu()}
  - Softplus {softplus()}
  - Mish {mish()}
  - Hardswish {hardswish()}

In [28]:
x1 = Node(5)
x2 = Node(3)
w1 = Node(4)
w2 = Node(2)
b = Node(7)
a = x1*w1
d = x2*w2
c = a + d
preactivation = c + b
activation = preactivation.tanh()
activation.backward()
print(f"Activation data : {activation.data}")
print(f"Activation gradient : {activation.gradient}")
print(f"Preactivation data : {preactivation.data}")
print(f"Preactivation gradient : {preactivation.gradient}")
print(f"x1w1 + x2w2 data : {c.data}")
print(f"x1w1 + x2w2 gradient : {c.gradient}")
print(f"x1w1 data : {a.data}")
print(f"x1w1 gradient : {a.gradient}")
print(f"x2w2 data : {d.data}")
print(f"x2w2 gradient : {d.gradient}")
print(f"x1 data : {x1.data}")
print(f"x1 gradient : {x1.gradient}")
print(f"x2 data : {x2.data}")
print(f"x2 gradient : {x2.gradient}")
print(f"w1 data : {w1.data}")
print(f"w1 gradient : {w1.gradient}")
print(f"w2 data : {w2.data}")
print(f"w2 gradient : {w2.gradient}")
print(f"b data : {b.data}")
print(f"b gradient : {b.gradient}")

Activation data : 1.0
Activation gradient : 1
Preactivation data : 33
Preactivation gradient : 0.0
x1w1 + x2w2 data : 26
x1w1 + x2w2 gradient : 0.0
x1w1 data : 20
x1w1 gradient : 0.0
x2w2 data : 6
x2w2 gradient : 0.0
x1 data : 5
x1 gradient : 0.0
x2 data : 3
x2 gradient : 0.0
w1 data : 4
w1 gradient : 0.0
w2 data : 2
w2 gradient : 0.0
b data : 7
b gradient : 0.0


In [29]:
# 1. Initialize leaf nodes with small values
x1 = Node(0.5)
x2 = Node(-0.3)
w1 = Node(1.0)
w2 = Node(-2.0)
b = Node(0.5)

# 2. Forward pass (Neuron computation)
w1x1 = x1 * w1
w2x2 = x2 * w2
w1x1_plus_w2x2 = w1x1 + w2x2
preactivation = w1x1_plus_w2x2 + b
activation = preactivation.tanh()

# 3. Backward pass
activation.backward()

# 4. Print results
print(f"Activation data : {activation.data}")
print(f"Activation gradient : {activation.gradient}")
print(f"Preactivation data : {preactivation.data}")
print(f"Preactivation gradient : {preactivation.gradient}")
print(f"w1x1 + w2x2 data : {w1x1_plus_w2x2.data}")
print(f"w1x1 + w2x2 gradient : {w1x1_plus_w2x2.gradient}")
print(f"x1w1 data : {w1x1.data}")
print(f"x1w1 gradient : {w1x1.gradient}")
print(f"x2w2 data : {w2x2.data}")
print(f"x2w2 gradient : {w2x2.gradient}")
print(f"x1 data : {x1.data}")
print(f"x1 gradient : {x1.gradient}")
print(f"x2 data : {x2.data}")
print(f"x2 gradient : {x2.gradient}")
print(f"w1 data : {w1.data}")
print(f"w1 gradient : {w1.gradient}")
print(f"w2 data : {w2.data}")
print(f"w2 gradient : {w2.gradient}")
print(f"b data : {b.data}")
print(f"b gradient : {b.gradient}")

Activation data : 0.9216685544064713
Activation gradient : 1
Preactivation data : 1.6
Preactivation gradient : 0.1505270758182854
w1x1 + w2x2 data : 1.1
w1x1 + w2x2 gradient : 0.1505270758182854
x1w1 data : 0.5
x1w1 gradient : 0.1505270758182854
x2w2 data : 0.6
x2w2 gradient : 0.1505270758182854
x1 data : 0.5
x1 gradient : 0.1505270758182854
x2 data : -0.3
x2 gradient : -0.3010541516365708
w1 data : 1.0
w1 gradient : 0.0752635379091427
w2 data : -2.0
w2 gradient : -0.04515812274548562
b data : 0.5
b gradient : 0.1505270758182854


In [ ]:
from node import Node # From node.py file import Node class
from utils import FormattedList # From utils.py file import FormattedList Class
class Neuron:
    INSTANCE_PARAMETERS = [
    "Inputs (inputs)",
    "Weight Initialization (weight_initialization)",
    "Pre-Activation (pre_activation)",
    "Activation (activation)",
    "Weight Initialization Param (weight_initialization_param)",
    "Pre-Activation Param (pre_activation_param)",
    "Activation Param (activation_param)",
    ]

    WEIGHT_INITIALIZATION_FUNCTIONS = [
    "Xavier Normal (xavier_norm)",
    "Xavier Uniform (xavier_uni)",
    "He Normal (he_norm)",
    "He Normal Negative Slope (he_norm_negative_slope)",
    "LeCun Normal (lecun_norm)",
    ]

    PRE_ACTIVATION_FUNCTIONS = [
    "Linear (linear)",
    "Dense (dense)",
    "MatMul (matmul)",
    "Bilinear (bilinear)",
    "Polynomial (polynomial)",
    "Max Pool (max_pool)",
    "Average Pool (avg_pool)",
    "Global Average Pool (global_avg_pool)",
    "Global Max Pool (global_max_pool)",
    "Stochastic Pool (stochastic_pool)",
    "Radial Basis (radial_basis)",
    "Cosine Similarity (cosine_similarity)",
    "Product Unit (product_unit)",
    "Mahalanobis Distance (mahalanobis_distance)",
    "Quaternion (quaternion)",
    "Extremum (extremum)",
    "Fuzzy Logic (fuzzy_logic)",
    ]

    ACTIVATION_FUNCTIONS = [
    "Tanh (tanh)",
    "ReLU (relu)",
    "Threshold ReLU (threshold_relu)",
    "PReLU (prelu)",
    "Leaky ReLU (leaky_relu)",
    "Sigmoid (sigmoid)",
    "SiLU (silu)",
    "ELU (elu)",
    "GELU (gelu)",
    "SELU (selu)",
    "Softplus (softplus)",
    "Mish (mish)",
    "Hardswish (hardswish)",
    "Softmax (softmax)",
    "Log Softmax (log_softmax)",
    "Gumbel Softmax (gumbel_softmax)",
    "Maxout (maxout)",
    "Sparsemax (sparsemax)",
    "Entmax (entmax)"
    ]

@classmethod
def _format(cls, mapping):
    result = []
    for name, items in mapping:
        result.append(f"{name}:")
        result.extend(f"  - {item}" for item in items)
    return FormattedList(result)

@classmethod
def get_parameters(cls):
    return cls._format([("Weight Initialization Functions", cls.INSTANCE_PARAMETERS)])

get_parameter = get_instance_parameter = get_instance_parameter = get_parameters

@classmethod
def get_weight_initializations(cls):
    return cls._format([("Weight Initialization Functions", cls.WEIGHT_INITIALIZATION_FUNCTIONS)])

get_weight_initialization = get_weight_initialization_function = get_weight_initialization_functions = get_weight_initializations

@classmethod
def get_pre_activations(cls):
    return cls._format([("Pre-Activation Functions", cls.PRE_ACTIVATION_FUNCTIONS)])

get_pre_activation = get_pre_activation_function = get_pre_activation_functions = get_pre_activations

@classmethod
def get_activations(cls):
    return cls._format([("Activation Functions", cls.ACTIVATION_FUNCTIONS)])

get_activation = get_activation_function = get_activation_functions = get_activations
    
def __init__(
        self, inputs=[], 
        weight_initialization='',
        pre_activation='linear',  
        activation='relu', 
        weight_initialization_param=None,
        pre_activation_param=None, 
        activation_param=None):

    self.weight_initialization = weight_initialization
    self.pre_activation = pre_activation
    self.activation = activation

    self.weight_initialization_param = weight_initialization_param or {}
    self.pre_activation_param = pre_activation_param or {} # Dictionary for multiple arguements. Example {"degree": 3, "other_param": value}
    self.activation_param = activation_param or {}

    if 'dim' in weight_initialization_param and 'degree' in pre_activation_param:
        if weight_initialization_param['dim'] != pre_activation_param['degree']:
            raise ValueError("Dimension and Degree must be same value.")


    self.inputs = inputs
    if isinstance(inputs, (int, float, Node)): # Returns true if inputs is int, float, or Node
        inputs = [inputs]
    
    self.inputs = []
    for item in inputs:
        if isinstance(item, Node):
            self.inputs.append(item)
        elif isinstance(item, (float, int)):
            node = Node(item, children=(), operator='input')
            self.inputs.append(node)
        else:
            raise ValueError(f"Unknown item datatype: {item}. Please input Node objects, integers, or floats.")

    self.weight_initializations = {
        'xavier_norm': lambda x: x.xavier_norm(**weight_initialization_param),
        'xavier_uni': lambda x: x.xavier_uni(**weight_initialization_param),
        'he_norm': lambda x: x.he_norm(**weight_initialization_param),
        'he_norm_negative_slope': lambda x: x.he_norm_negative_slope(**weight_initialization_param),
        'lecun_norm': lambda x: x.lecun_norm(**weight_initialization_param),
    }
    if weight_initialization == '':
        self.weight_initialization_function = None
    elif weight_initialization not in self.weight_initializations:
        raise ValueError(f"Unknown weight initialization '{weight_initialization}'. Choose from: {get_weight_initializations}")
    else:
        self.weight_initialization_function = self.weight_initializations[weight_initialization]
    
    self.pre_activations = {
    # linear formula
    'linear': lambda x: x.linear(**pre_activation_param),
    'dense': lambda x: x.linear(**pre_activation_param),
    # matrix multiplication formula
    'matmul': lambda x: x.matmul(**pre_activation_param),
    'mat_mul': lambda x: x.matmul(**pre_activation_param),
    # bilinear layer
    'bilinear': lambda x: x.bilinear(**pre_activation_param),
    'bilinear_layer': lambda x: x.bilinear(**pre_activation_param),
    # polynomial combination
    'polynomial': lambda x: x.polynomial(**pre_activation_param),
    'polynomial_combination': lambda x: x.polynomial(**pre_activation_param),
    # max pooling
    'max_pool': lambda x: x.max_pool(**pre_activation_param),
    'maximum_pool': lambda x: x.max_pool(**pre_activation_param),
    # average pooling
    'avg_pool': lambda x: x.avg_pool(**pre_activation_param),
    'average_pool': lambda x: x.avg_pool(**pre_activation_param),
    # global average pooling
    'global_avg_pool': lambda x: x.global_avg_pool(**pre_activation_param),
    'global_average_pool': lambda x: x.global_avg_pool(**pre_activation_param),
    # global max pooling
    'global_max_pool': lambda x: x.global_max_pool(**pre_activation_param),
    'global_maximum_pool': lambda x: x.global_max_pool(**pre_activation_param),
    # stochastic pooling
    'stochastic_pool': lambda x: x.stochastic_pool(**pre_activation_param),
    'stoch_pool': lambda x: x.stochastic_pool(**pre_activation_param),
    # radial basis
    'radial_basis': lambda x: x.radial_basis(**pre_activation_param),
    # cosine similarity
    'cosine_similarity': lambda x: x.cosine_similarity(**pre_activation_param),
    # product unit
    'product_unit': lambda x: x.product_unit(**pre_activation_param),
    # mahalanobis distance
    'mahalanobis_distance': lambda x: x.mahalanobis_distance(**pre_activation_param),
    # quaternion
    'quaternion': lambda x: x.quaternion(**pre_activation_param),
    # extremum
    'extremum': lambda x: x.extremum(**pre_activation_param),
    # fuzzy logic
    'fuzzy_logic': lambda x: x.fuzzy_logic(**pre_activation_param),
    }

    if pre_activation not in self.pre_activations:
        raise ValueError(f"Unknown pre-activation '{pre_activation}'. Choose from: {get_pre_activations}")
    else:
        self.pre_activation_function = self.pre_activations[pre_activation]

    # Access pre-activation function in dictionary using pre_activation arguement as index, call pre-activation function by writing {Neuron Object}.pre_activation_function()

    self.activations = {
    'tanh': lambda x: x.tanh(**activation_param),
    'relu': lambda x: x.relu(**activation_param),
    'threshold_relu': lambda x: x.threshold_relu(**activation_param),
    'thresholdrelu': lambda x: x.threshold_relu(**activation_param), # Alternate Name
    'prelu': lambda x: x.prelu(**activation_param),
    'leaky_relu': lambda x: x.leaky_relu(**activation_param),
    'leakyrelu': lambda x: x.leaky_relu(**activation_param), # Alternate name
    'sigmoid': lambda x: x.sigmoid(**activation_param), 
    'silu': lambda x: x.silu(**activation_param), 
    'elu': lambda x: x.elu(**activation_param),
    'gelu': lambda x: x.gelu(**activation_param),
    'selu': lambda x: x.selu(**activation_param),
    'softplus': lambda x: x.softplus(**activation_param),
    'mish': lambda x: x.mish(**activation_param),
    'hardswish': lambda x: x.hardswish(**activation_param), # Activation functions from here to above are element-wise activations
    'softmax': lambda x: x.softmax(**activation_param), # Activation functions from here to below are joint activations (pre activation combined with activation)
    'log_softmax': lambda x: x.log_softmax(**activation_param),
    'logsoftmax': lambda x: x.log_softmax(**activation_param), # Alternate Name
    'gumbel_softmax': lambda x: x.gumbell_softmax(**activation_param),
    'gumbelsoftmax': lambda x: x.gumbell_softmax(**activation_param), # Alternate name
    'maxout': lambda x: x.maxout(**activation_param),
    'sparsemax': lambda x: x.sparsemax(**activation_param),
    'entmax': lambda x: x.entmax(**activation_param),
    }

    if activation not in self.activations:
        raise ValueError(f"Unknown activation '{activation}'. Choose from: {get_activations}")
    else:
        self.activation_function = self.activations[activation] 

    # Access activation function in dictionary using activation arguement as index, call activation function by writing {Node/Neuron Object}.activation_function()

    self.weights = [] # Weights list for each input data used in pre activation
    
    self.z = self.pre_activation_function() # Node of pre activation function

    self.data = self.z.activation_function() # Node Output(s)  of the activation function 

def __repr__(self):
    return f"Neuron(data={self.data})"

# Copying Gauss Function from Python to manually turn update=False mu and z, for he_negative_slope when activation is prelu, keep connection with neg_slope so it can be learnable
def gauss(self, mu=0.0, sigma=1.0):
    _sqrt = math.sqrt
    _log = math.log
    _cos = math.cos
    _sin = math.sin
    TWOPI = math.pi * 2.0
    random = self.random
    z = self.gauss_next
    self.gauss_next = None
    if z is None:
        x2pi = random() * TWOPI
        g2rad = _sqrt(-2.0 * _log(1.0 - random()))
        z = _cos(x2pi) * g2rad
        self.gauss_next = _sin(x2pi) * g2rad

    return Node(mu, update=False) + Node(z, update=False) * sigma

# Dimension Recursion Function for weight initialization functions
def length_dimension(dim, length, fill):
    if dim <= 0:
        return fill

    if dim == 1:
        return [fill] * length
    
    return [
        length_dimension(dim - 1, length, fill)
        for _ in range(length)
    ]

# Weight Initialization Functions
def xavier_norm(self, dim=1):
    scaling = 1
    if dim > 2:
        scaling = dim
    std_dev = (2/(len(self.inputs)**scaling + 1))**0.5
    self.weights = length_dimension(dim, len(self.inputs), Node(random.gauss(0.0, std_dev), operator ='weights', update=True))

def xavier_uni(self, dim=1):
    limit = (6/len(self.inputs))**0.5
    self.weights = length_dimension(dim, len(self.inputs), Node(np.random.uniform(-limit, limit), operator='weights', update=True))

def he_norm(self, dim=1):
    scaling = 1
    if dim > 2:
        scaling = dim
    std_dev = (2/(len(self.inputs)**scaling))**0.5
    self.weights = length_dimension(dim, len(self.inputs), Node(random.gauss(0.0, std_dev), operator ='weights', update=True))

def he_norm_negative_slope(self, neg_slope=0.01, dim=1):
    scaling = 1
    if dim > 2:
        scaling = dim
    if self.activation == 'prelu':
        neg_slope = 0.25
        std_dev = (
            Node(2, children=(), operator="", update=False)
            / (
                Node(len(self.inputs), children=(), operator="", update=False)
                ** Node(scaling, children=(), operator="", update=False)
                * (
                    Node(1, children=(), operator="", update=False)
                    + Node(
                        neg_slope, children=(), operator="negative slope", update=True
                    )
                    ** Node(2, children=(), operator="", update=False)
                )
            )
        ) ** Node(0.5, children=(), operator="", update=False)
        self.weights = length_dimension(dim, len(self.inputs), Node(self.gauss(0.0, std_dev), operator ='weights', update=True))
    else:
        std_dev = (2/((len(self.inputs)**scaling)*(1 + neg_slope**2)))**0.5
        self.weights = length_dimension(dim, len(self.inputs), Node(random.gauss(0.0, std_dev), operator ='weights', update=True))

def lecun_norm(self, dim=1):
    scaling = 1
    if dim > 2:
        scaling = dim
    std_dev = (1/(len(self.inputs)**scaling))**0.5
    self.weights = length_dimension(dim, len(self.inputs), Node(random.gauss(0.0, std_dev), operator ='weights', update=True))

# Pre Activation Functions

def linear(self):
    if self.weight_initialization:
        self.weight_initialization_function()
    elif self.activation in ['tanh', 'sigmoid']:
        self.xavier_norm()
    elif self.activation in [
    'relu', 
    'threshold_relu', 'thresholdrelu', 
    'silu', 
    'elu', 
    'gelu', 
    'softplus', 
    'mish', 
    'hardswish'
    ]:
        self.he_norm()
    elif self.activation in ['leaky_relu', 'leakyrelu', 'prelu']:
        self.he_norm_negative_slope()
    elif self.activation in ['selu']:
        self.lecun_norm()
    weighed_sum = Node(0, children=(), operator='bias')
    for i in range(len(self.inputs)):
        weighed_sum += self.inputs[i] * self.weights[i]
    return weighed_sum

def bilinear(self):
    dimensions = np.array(self.inputs).ndim
    if dimensions < 0 or dimensions > 2:
        raise ValueError(f"Invalid input dimension: {dimensions}. Please input 1 or 2 dimensional input.")
    dim = dimensions if dimensions == 1 else 4
    if self.weight_initialization:
        self.weight_initialization_function()
    elif self.activation in ['tanh', 'sigmoid']:
        self.xavier_uni(dim = dim)
    elif self.activation in ['softplus']:
        self.xavier_norm(dim = dim)
    elif self.activation in [
    'relu', 
    'threshold_relu', 'thresholdrelu', 
    'silu', 
    'elu', 
    'gelu', 
    'mish', 
    'hardswish'
    ]:
        self.he_norm(dim=dim)
    elif self.activation in ['leaky_relu', 'leakyrelu', 'prelu']:
        self.he_norm_negative_slope(dim = dim)
    elif self.activation in ['selu']:
        self.lecun_norm(dim = dim)
    weighed_sum = Node(0, children=(), operator='bias')
    if dimensions == 1:
        for row in range(len(self.inputs)):
            for col in range(len(self.inputs)):
                weighed_sum += self.inputs[row] * self.inputs[col] * self.weights[row][col]
        return weighed_sum
    else:
        for row_one in range(len(self.inputs)):
            for col_one in range(len(self.inputs[0])):
                for row_two in range(len(self.inputs)):
                    for col_two in range(len(self.inputs[0])):
                        weighed_sum += self.inputs[row_one][col_one] * self.inputs[row_two][col_two] * self.weights[row_one][col_one][row_two][col_two]
        return weighed_sum

# To see polynomial caombination in for loops before making recursion code that can handle all degrees
def polynomial_manual(self, degree = 2):
    if self.weight_initialization:
        self.weight_initialization_function()
    if degree == 1:
        return self.linear()
    elif degree == 2:
        weighed_sum = Node(0, children=(), operator='bias')
        for i in range(len(self.inputs)):
            weighed_sum += self.inputs[i] * self.weights[i][i]
            for j in range(len(self.inputs)):
                weighed_sum += self.inputs[i] * self.inputs[j] * self.weights[i][j]
        return weighed_sum
    elif degree == 3:
        weighed_sum = Node(0, children=(), operator='bias')
        for i in range(len(self.inputs)):
            weighed_sum += self.inputs[i] * self.weights[i][i][i]
            for j in range(len(self.inputs)):
                weighed_sum += self.inputs[i] * self.inputs[j] * self.weights[i][j][j]
                for k in range(len(self.inputs)):
                    weighed_sum += self.inputs[i]*self.inputs[j]*self.inputs[k] * self.weights[i][j][k]
        return weighed_sum
    elif degree == 4:
        weighed_sum = Node(0, children=(), operator='bias')
        for i in range(len(self.inputs)):
            weighed_sum += self.inputs[i] * self.weights[i][i][i][i]
            for j in range(len(self.inputs)):
                weighed_sum += self.inputs[i] * self.inputs[j] * self.weights[i][j][j][j]
                for k in range(len(self.inputs)):
                    weighed_sum += self.inputs[i]*self.inputs[j]*self.inputs[k] * self.weights[i][j][k][k]
                    for l in range(len(self.inputs)):
                        weighed_sum += self.inputs[i]*self.inputs[j]*self.inputs[k]*self.inputs[l]*self.weights[i][j][k][l]
        return weighed_sum
    elif degree == 5:
        weighed_sum = Node(0, children=(), operator='bias')
        for i in range(len(self.inputs)):
            weighed_sum += self.inputs[i] * self.weights[i][i][i][i][i] # Amount of inputs powered by Number of different for loop variables equals amount of total iterations including nested
            for j in range(len(self.inputs)):
                weighed_sum += self.inputs[i] * self.inputs[j] * self.weights[i][j][j][j][j]
                for k in range(len(self.inputs)):
                    weighed_sum += self.inputs[i]*self.inputs[j]*self.inputs[k] * self.weights[i][j][k][k][k]
                    for l in range(len(self.inputs)):
                        weighed_sum += self.inputs[i]*self.inputs[j]*self.inputs[k]*self.input[l]*self.weights[i][j][k][l][l]
                        for m in range(len(self.inputs)):
                            weighed_sum += self.inputs[i]*self.inputs[j]*self.inputs[k]*self.input[l]*self.input[m]*self.weights[i][j][k][l][m]
        return weighed_sum
    
def polynomial(self, degree=2):
    if degree <= 0:
        raise ValueError(f"Degree value: {degree}. Please select a degree value greater than 0 for the polynomial combination.")
    elif degree == 1:
        if self.weight_initialization:
            self.weight_initialization_function()
        elif self.activation in ['tanh', 'sigmoid']:
            self.xavier_norm()
        elif self.activation in [
        'relu', 
        'threshold_relu', 'thresholdrelu', 
        'silu', 
        'elu', 
        'gelu', 
        'softplus', 
        'mish', 
        'hardswish'
        ]:
            self.he_norm()
        elif self.activation in ['leaky_relu', 'leakyrelu', 'prelu']:
            self.he_norm_negative_slope()
        elif self.activation in ['selu']:
            self.lecun_norm()
    elif degree == 2:
        if self.weight_initialization:
            self.weight_initialization_function()
        elif self.activation in ['tanh', 'sigmoid']:
            self.xavier_uni(dim = 2)
        elif self.activation in ['softplus']:
            self.xavier_norm(dim = 2)
        elif self.activation in [
        'relu', 
        'threshold_relu', 'thresholdrelu', 
        'silu', 
        'elu', 
        'gelu', 
        'mish', 
        'hardswish'
        ]:
            self.he_norm(dim=2)
        elif self.activation in ['leaky_relu', 'leakyrelu', 'prelu']:
            self.he_norm_negative_slope(dim = 2)
        elif self.activation in ['selu']:
            self.lecun_norm(dim = 2)
    else:
        if self.activation in ['tanh', 'sigmoid']:
            self.xavier_uni(dim = degree)
        elif self.activation in ['softplus']:
            self.xavier_norm(dim = degree)
        elif self.activation in [
        'relu', 
        'threshold_relu', 'thresholdrelu', 
        'silu', 
        'elu', 
        'gelu', 
        'mish', 
        'hardswish'
        ]:
            self.he_norm(dim=degree)
        elif self.activation in ['leaky_relu', 'leakyrelu', 'prelu']:
            self.he_norm_negative_slope(dim = degree)
        elif self.activation in ['selu']:
            self.lecun_norm(dim = degree)

    loople = []
    weighed_sum = Node(0, children=(), operator='bias')
    for _ in range(degree):
        loople += [0]
    
    def poly(degree = 2, level = 1,loople=[]):
       loople = list(loople)
       for _ in range(len(self.inputs)):
            w = self.weights
            input_mul = 1
            inputs = []

            for i in range(degree):
                w = w[loople[i]]

            for j in range(level):
                inputs.append(self.inputs[loople[j]])
                input_mul *= inputs[j]

            nonlocal weighed_sum # To call weighed_sum in the function outsided of this one, eventualyy reaching polynomial fucntion's weighed_sum
            weighed_sum += w * input_mul

            loople[level - 1] += 1
            for k in range(level, degree):
                loople[k] = 0
    
            if level < degree:
                poly(degree = degree, level = level + 1, loople = loople)

            if loople[level - 1] < len(self.inputs):
                for l in range(level, degree):
                    loople[l] = loople[level - 1]

    poly(degree = degree, level =1, loople=loople)
    return weighed_sum

def max_pool(self):
    if np.array(self.inputs).ndim > 1:
        return global_max_pool()
    max = Node(max(self.inputs).data, children=(self.inputs), operator='z')
    def max_pool_grad():
        for input in self.inputs:
            if input.data == max.data:
                input.gradient += max.gradient
            else:
                input.gradient += 0
    max._backward = max_pool_grad
    return max

def avg_pool(self):
    if np.array(self.inputs).ndim > 1:
        return global_avg_pool()
    sum = 0
    for i in range(len(self.inputs)):
        sum += self.inputs[i]
    return (1/len(self.inputs)) * sum

def global_avg_pool_manual(self):
    dimension = np.array(self.inputs)
    if dimension == 1:
        avg_pool()
    elif dimension == 2:
        height = len(self.inputs)
        width = len(self.inputs[0])
        sum = 0
        for y in range(height):
            for x in range(width):
                sum += self.inputs[y][x]
        return (1/(height * width)) * sum
    elif dimension == 3:
        height = len(self.inputs)
        width = len(self.inputs[0])
        depth = len(self.inputs[0][0])
        sum = 0
        for y in range(height):
            for x in range(width):
                for z in range(depth):
                    sum += self.inputs[y][x][z]
        return (1/(height * width * depth)) * sum

def global_avg_pool(self):
    dimension = np.array(self.inputs).ndim
    measure = self.inputs
    dims_mul = 1
    weighed_sum = 0
    dims =[]
    if dimension > 1:
        for i in range(dimension):
            dims.append(len(measure))
            dims_mul *= 1/len(measure)
            measure = self.inputs[0]
    else:
        return avg_pool()
    
    loople = []
    for _ in range(dimension):
        loople += [0]

    
    def global_avg(level = 1):
        level = level
        dim_no = dims[level - 1]
        new_level = level + 1
        if level == dimension + 1:
            input_data = self.inputs
            for j in range(dimension):
                input_data = input_data[loople[j]]
            nonlocal weighed_sum 
            weighed_sum += input_data
        else:
            order = level - 1
            for i in range(dim_no):
                loople[order] = i
                global_avg(level = new_level)
    global_avg()
    z = dims_mul * weighed_sum
    return z

# Flatten any dimensional array into 1D
def flatten(self):
        for item in self.inputs:
            if isinstance(item, list):
                yield from flatten(item) # yield is a function generator keyword, from ketword ensures all items are contained and goes through its own recursion
            else:
                yield item

def global_max_pool(self):
    if np.array(self.inputs).ndim == 1:
        return max_pool
    flattened = list(self.flatten(self.inputs))
    max = Node(max(flattened).data, children = (flattened), operator='z')
    def global_max_pool_grad():
        for item in flattened:
            if item.data == max.data:
                item.gradient += max.gradient
            else:
                item.gradient += 0
    max._backward = global_max_pool_grad
    return max
            
def stochastic_pool(self):
    flattened = list(self.flatten(self.inputs))
    data = [node.data for node in flattened]
    total_sum = sum(data)
    prob = []
    for item in data:
        prob.append(item/total_sum)
    index = random.choices(range(len(prob)), weights=prob)[0] # We add [0] because it returns a list and our default k value is 1 meaning it will return 1 index value
    stoch = Node(flattened[index].data, children = (flattened), operator = 'z')
    def stochastic_pool_grad():
        for i, item in enumerate(flattened):
            if i == index:
                item.gradient += stoch.gradient
            else:
                item.gradient += 0
    stoch._backward = stochastic_pool_grad
    return stoch

def radial_basis(self):
    dimension = np.array(self.inputs).ndim
    if self.weight_initialization:
        self.weight_initialization_function()
    elif self.activation in ['tanh', 'sigmoid']:
        self.xavier_norm(dim = dimension)
    elif self.activation in [
    'relu', 
    'threshold_relu', 'thresholdrelu', 
    'silu', 
    'elu', 
    'gelu', 
    'softplus', 
    'mish', 
    'hardswish'
    ]:
        self.he_norm(dim = dimension)
    elif self.activation in ['leaky_relu', 'leakyrelu', 'prelu']:
        self.he_norm_negative_slope(dim = dimension)
    elif self.activation in ['selu']:
        self.lecun_norm(dim = dimension)
    
    measure = self.inputs
    weighed_sum = 0
    dims =[]
    loople = []

    for i in range(dimension):
        dims.append(len(measure))
        if dimension > 1:
            measure = self.inputs[0]

    for _ in range(dimension):
        loople += [0]

    def rbf(level = 1):
        level = level
        dim_no = dims[level - 1]
        new_level = level + 1
        if level == dimension + 1:
            input_data = self.inputs
            weight_data = self.weights
            for j in range(dimension):
                input_data = input_data[loople[j]]
                weight_data = weight_data[loople[j]]
            nonlocal weighed_sum 
            weighed_sum += (input_data - weight_data)**2
        else:
            order = level - 1
            for i in range(dim_no):
                loople[order] = i
                rbf(level = new_level)
    rbf()
    z = weighed_sum**0.5
    return z



    





    

# Backward Function





In [4]:
import inspect
import random

# Force print line-by-line to prevent notebook truncation
for line in inspect.getsource(random.gauss).splitlines():
    print(line)

    def gauss(self, mu=0.0, sigma=1.0):
        """Gaussian distribution.

        mu is the mean, and sigma is the standard deviation.  This is
        slightly faster than the normalvariate() function.

        Not thread-safe without a lock around calls.

        """
        # When x and y are two variables from [0, 1), uniformly
        # distributed, then
        #
        #    cos(2*pi*x)*sqrt(-2*log(1-y))
        #    sin(2*pi*x)*sqrt(-2*log(1-y))
        #
        # are two *independent* variables with normal distribution
        # (mu = 0, sigma = 1).
        # (Lambert Meertens)
        # (corrected version; bug discovered by Mike Miller, fixed by LM)

        # Multithreading note: When two threads call this function
        # simultaneously, it is possible that they will receive the
        # same return value.  The window is very small though.  To
        # avoid this, you have to use a lock around all calls.  (I
        # didn't want to slow this down in the serial case b

In [ ]:
from node import Node
from neuron import Neuron
from utils import FormattedList

class Layer:
    NORMALIZATION_FUNCTIONS = [
    "Batch Normalization (batch)",
    "Layer Normalization (layer)",
    "RMS Normalization (rms)",
    "Instance Normalization (instance)",
    "Group Normalization (group)",
    "Weight Normalization (weight)",
    "Spectral Normalization (spectral)",
    "Local Response Normalization (local_response)"
    ]

    @classmethod
    def get_normalizations(cls):
        return cls._format([("Activation Functions", cls.PRE_ACTIVATION_FUNCTIONS)])

    get_normalization = get_normalization_function = get_normalization_functions = get_normalizations

    def __init__(
            self,
            normalization='None', 
            normalization_param=None,
            ):
        self.normalization_param = normalization_param or {} # 'or {}' to handle when no arguement

        self.normalizations = {
        # batch
        'batch': lambda x: x.batch(**normalization_param),
        'batch_normalization': lambda x: x.batch(**normalization_param),
        
        # layer
        'layer': lambda x: x.layer(**normalization_param),
        'layer_normalization': lambda x: x.layer(**normalization_param),
        
        # root mean squared
        'rms': lambda x: x.rms(**normalization_param),
        'rms_normalization': lambda x: x.rms(**normalization_param),
        
        # instance
        'instance': lambda x: x.instance(**normalization_param),
        'instance_normalization': lambda x: x.instance(**normalization_param),
        
        # group
        'group': lambda x: x.group(**normalization_param),
        'group_normalization': lambda x: x.group(**normalization_param),
        
        # weight
        'weight': lambda x: x.weight(**normalization_param),
        'weight_normalization': lambda x: x.weight(**normalization_param),
        
        # spectral
        'spectral': lambda x: x.spectral(**normalization_param),
        'spectral_normalization': lambda x: x.spectral(**normalization_param),
        
        # local response
        'local_response': lambda x: x.local_response(**normalization_param),
        'local_response_normalization': lambda x: x.local_response(**normalization_param),
        'lrn': lambda x: x.local_response(**normalization_param),
        }
        if normalization not in self.normalizations:
            raise ValueError(f"Unknown normalization '{normalization}'. Choose from: {get_normalizations}")
        else:
            self.normalization_function = self.normalizations[normalization]

    # Access normalization function in dictionary using normalization arguement as index, call normalization function by writing {Neuron Object}.normalization_function()